# 09 - Inference server for the chat app
Serves the trained models to the chat app in `app/frontend` (step-by-step guide in `app/README.md`).
The same notebook runs on two Kaggle accounts: shard `a` serves the 4B models, shard `b` the 2B models.
Base models are downloaded from Hugging Face; the LoRA adapters and eval outputs come from the private dataset `selfrefine-adapters`.
Started with `app/kaggle/push.ps1` (a background "Save & Run All" run on each account; Kaggle stops it after 12 hours).

Request path: browser -> Vercel `/api/generate` -> ngrok -> `router.py` (port 9000) -> vLLM (one server per T4, ports 8000 and 8001).

In [ ]:
!pip uninstall -y torchaudio torchao # newer Kaggle image: old torchaudio breaks "import datasets" after vllm upgrades torch
!pip install -U transformers==5.18.0 peft==0.21.1 vllm math-verify datasets fastapi uvicorn httpx pyngrok

In [ ]:
import sys
import torch
import vllm
print("Python:", sys.version)
print("Torch:", torch.__version__)
print("vLLM:", vllm.__version__)
print("GPUs:", torch.cuda.device_count())

In [ ]:
import glob
import json
import os
# Settings come from server_config.json in the private "selfrefine-server-config" dataset (written by app/kaggle/push.ps1),
# or from Kaggle secrets with the same names when the notebook is run by hand.
config_files=glob.glob('/kaggle/input/**/server_config.json',recursive=True)
if config_files:
    config=json.load(open(config_files[0]))
else:
    from kaggle_secrets import UserSecretsClient
    secrets=UserSecretsClient()
    config={name:secrets.get_secret(name) for name in ['APP_API_KEY','NGROK_AUTHTOKEN','NGROK_DOMAIN','SHARD']}
api_key=config['APP_API_KEY']          # same value as APP_API_KEY on Vercel
ngrok_token=config['NGROK_AUTHTOKEN']
ngrok_domain=config['NGROK_DOMAIN']    # e.g. something.ngrok-free.app (fixed, so the Vercel setting never changes)
shard=config['SHARD']                  # 'a' = 4B models (first Kaggle account), 'b' = 2B models (second account)
mode=config.get('MODE','lora')         # 'lora' = adapters on top of one base model | 'merged' = fallback if vLLM rejects the adapters
check_n=20                             # MATH-500 problems compared with the reported eval outputs (0 = skip the check)
print("Shard:", shard, "| mode:", mode)

## Models
Ids match `app/frontend/lib/models.ts`. `eval` is the file in `experiments/08_eval_rp/` with the reported outputs.

In [ ]:
MODELS={
    '4b-base':       {'base':'4b','adapter':None,                     'kind':'base',  'eval':'qwen4b_math500_baseline_rp1.1.json'},
    '4b-cot':        {'base':'4b','adapter':'qwen4b-cot_4b-final',    'kind':'cot',   'eval':'eval_cot_4b_rp1.1.json'},
    '4b-review':     {'base':'4b','adapter':'qwen4b-review_4b-final', 'kind':'review','eval':'eval_review_4b_rp1.1.json'},
    '2b-base':       {'base':'2b','adapter':None,                     'kind':'base',  'eval':'qwen2b_math500_baseline_rp1.1.json'},
    '2b-cot':        {'base':'2b','adapter':'qwen2b-cot-final',       'kind':'cot',   'eval':'eval_cot_rp1.1.json'},
    '2b-refine':     {'base':'2b','adapter':'qwen2b-refine-final',    'kind':'refine','eval':'eval_refine_rp1.1.json'},
    '2b-review-9b':  {'base':'2b','adapter':'qwen2b-review-final',    'kind':'review','eval':'eval_review_rp1.1.json'},
    '2b-review-27b': {'base':'2b','adapter':'qwen2b-review_27b-final','kind':'review','eval':'eval_review_27b_rp1.1.json'},
}
BASES={'4b':'Qwen/Qwen3.5-4B-Base','2b':'Qwen/Qwen3.5-2B-Base'}

## Which models run where
Two Kaggle accounts with two T4s each. A base model and its adapters must be on the same GPU, so the split is by base model.
Each GPU holds its own copy of the base, so compare mode (3 answers at once) runs on two GPUs instead of one.
`merged` fallback: one merged model per GPU, chosen so compare mode (4B base, CoT, self-refine) still works.

In [ ]:
PLANS={
    'lora':{
        'a':[('4b',['4b-review']), ('4b',['4b-base','4b-cot'])],
        'b':[('2b',['2b-review-27b','2b-review-9b']), ('2b',['2b-base','2b-cot','2b-refine'])],
    },
    'merged':{
        'a':[('4b',['4b-review']), ('4b',['4b-cot'])],
        'b':[('4b',['4b-base']), ('2b',['2b-review-27b'])],
    },
}
plan=PLANS[mode][shard]
for gpu,(size,models) in enumerate(plan):
    print(f'GPU {gpu}: {BASES[size]} ->', models)

In [ ]:
adapter_dirs={os.path.basename(os.path.dirname(p)):os.path.dirname(p) for p in glob.glob('/kaggle/input/**/adapter_config.json',recursive=True)}
eval_files={os.path.basename(p):p for p in glob.glob('/kaggle/input/**/*_rp1.1.json',recursive=True)}
print("Adapters:", sorted(adapter_dirs))
print("Eval files:", sorted(eval_files))

## Start vLLM
`lora`: the base model loads once per GPU and each adapter is a LoRA module on it.
`merged`: same merge as notebook 07, the merged model is served on its own.

In [ ]:
import gc
servers=[] # vLLM processes to start, one per GPU
routes={}  # model id -> where the router sends it
for gpu,(size,models) in enumerate(plan):
    port=8000+gpu
    path=BASES[size]
    name=f'{size}-base'
    loras={}
    for m in models:
        c=MODELS[m]
        if c['adapter'] and mode=='lora':
            assert c['adapter'] in adapter_dirs, f"Adapter folder {c['adapter']} not found in /kaggle/input"
            loras[m]=adapter_dirs[c['adapter']]
        elif c['adapter']: # merged: same merge as notebook 07
            from transformers import AutoTokenizer, AutoModelForCausalLM
            from peft import PeftModel
            base_model=AutoModelForCausalLM.from_pretrained(path,dtype=torch.float16)
            model=PeftModel.from_pretrained(base_model,adapter_dirs[c['adapter']]).merge_and_unload()
            path=f'/tmp/merged-{m}' # /tmp: two merged 4B models (~17 GB) would not fit the 20 GB /kaggle/working limit
            model.save_pretrained(path)
            AutoTokenizer.from_pretrained(BASES[size]).save_pretrained(path)
            name=m
            del model, base_model
            gc.collect()
        # a model without an adapter is the base itself; with LoRA it is the adapter's name on this server
        routes[m]={'port':port,'served':m if m in loras else name,'kind':c['kind']}
    servers.append({'gpu':gpu,'port':port,'model':path,'name':name,'loras':loras})
print(json.dumps(servers,indent=1))
print("Models served:", list(routes))

In [ ]:
import subprocess
procs=[]
for s in servers:
    cmd=['vllm','serve',s['model'],
         '--served-model-name',s['name'],
         '--port',str(s['port']),
         '--dtype','float16',
         '--max-model-len','4096',
         '--gpu-memory-utilization','0.85',
         '--enforce-eager'] # same engine settings as notebooks 01 and 07
    if s['model'].startswith('Qwen/'): # the base repos include a vision part: switch image/video input off, as in notebook 01
        cmd+=['--limit-mm-per-prompt',json.dumps({'image':0,'video':0})]
    if s['loras']:
        cmd+=['--enable-lora','--max-lora-rank','16','--max-loras',str(len(s['loras'])),
              '--lora-modules']+[f'{name}={path}' for name,path in s['loras'].items()]
    env={**os.environ,'CUDA_VISIBLE_DEVICES':str(s['gpu'])}
    log=open(f"/kaggle/working/vllm_{s['port']}.log",'w')
    procs.append(subprocess.Popen(cmd,env=env,stdout=log,stderr=subprocess.STDOUT))
    print("Started:", ' '.join(cmd))

In [ ]:
import time
import requests
def wait_ready(port: int, proc, timeout: int=1500)->bool:
    start=time.time()
    while time.time()-start<timeout:
        if proc.poll() is not None: # process died
            return False
        try:
            if requests.get(f'http://localhost:{port}/v1/models',timeout=3).ok:
                return True
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(10)
    return False

for s,p in zip(servers,procs):
    ok=wait_ready(s['port'],p)
    print(f"GPU {s['gpu']} {s['name']}:",'ready' if ok else 'FAILED')
    if not ok:
        print(open(f"/kaggle/working/vllm_{s['port']}.log").read()[-6000:])
        raise RuntimeError('vLLM did not start, see the log above. If it is a LoRA error, set MODE to merged in app/kaggle/accounts.json')
    print([m['id'] for m in requests.get(f"http://localhost:{s['port']}/v1/models").json()['data']])

## Router
Small FastAPI app between the chat app and vLLM: checks the API key, builds the exact eval prompt, streams tokens, and at the end
grades and splits the answer with the same functions as notebook 07.

In [ ]:
%%writefile /kaggle/working/router.py
# Written by notebooks/09_inference_server.ipynb. Runs on port 9000 behind ngrok.
import asyncio
import json
import os
import secrets
import time
import httpx
from fastapi import Depends, FastAPI, Header, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from math_verify import parse, verify

API_KEY=os.environ['APP_API_KEY']
ROUTES=json.load(open(os.environ.get('ROUTES_FILE','/kaggle/working/routes.json'))) # model id -> {port, served, kind}
MAX_QUESTION_CHARS=3000   # longest training question is 2,222 characters
BUSY=asyncio.Semaphore(8) # at most 8 answers at once, others wait their turn

def format_prompt(kind: str, question: str)->str:
    if kind=='base': # notebook 01
        result=''
        result+='Question: '+question+'\n'
        result+='Please reason step by step, and put your final answer within \\boxed{}.'+'\n'
        result+='Answer:'
        return result
    return '[Question] '+question+'\n' # notebook 07

def sampling(kind: str)->dict:
    # greedy with repetition_penalty 1.1, the settings behind the README numbers
    if kind=='base': # notebook 01
        return {'temperature':0.0,'repetition_penalty':1.1,'max_tokens':2048,
                'stop':["\n\nQuestion:", "Question:", "<|endoftext|>", "<|im_end|>"]}
    return {'temperature':0.0,'repetition_penalty':1.1,'max_tokens':3072} # notebook 07

def extract_boxed(text: str):
    start=text.rfind('\\boxed{')
    if start==-1:
        return None
    i=start+len('\\boxed{')
    depth=1
    while i<len(text) and depth>0:
        if text[i]=='{':
            depth+=1
        elif text[i]=='}':
            depth-=1
        i+=1
    if depth!=0:
        return None
    return text[start+len('\\boxed{'):i-1]

def is_correct(answer, gold: str)->bool:
    if answer is None:
        return False
    try:
        return bool(verify(parse('$'+gold+'$'),parse('\\boxed{'+answer+'}')))
    except Exception:
        return False

def same_answer(a: str, b: str)->bool:
    return a.strip()==b.strip() or is_correct(a,b)

def get_first_answer(text: str):
    if '[Answer 1]' not in text:
        return None
    part=text.split('[Answer 1]')[1].split('[Answer 2]')[0].split('[Final answer]')[0]
    return extract_boxed(part)

def get_final_answer(text: str):
    if '[Final answer]' not in text:
        return None
    return text.split('[Final answer]')[-1].strip()

def final_answer(kind: str, text: str):
    return extract_boxed(text) if kind=='base' else get_final_answer(text)

def event(data: dict)->str:
    return 'data: '+json.dumps(data)+'\n\n'

app=FastAPI()

def check_key(authorization: str|None=Header(default=None)):
    if not authorization or not secrets.compare_digest(authorization,'Bearer '+API_KEY):
        raise HTTPException(401,'Wrong or missing API key')

class GenerateRequest(BaseModel):
    model: str
    question: str
    reference: str|None=None # known answer, only for the examples

@app.get('/health')
def health():
    return {'ok':True,'models':list(ROUTES)}

@app.post('/generate',dependencies=[Depends(check_key)])
async def generate(req: GenerateRequest):
    if req.model not in ROUTES:
        raise HTTPException(404,f'Model {req.model} is not loaded on the server')
    if not req.question.strip() or len(req.question)>MAX_QUESTION_CHARS:
        raise HTTPException(422,f'The question must be 1 to {MAX_QUESTION_CHARS} characters')
    route=ROUTES[req.model]
    body={'model':route['served'],'prompt':format_prompt(route['kind'],req.question),**sampling(route['kind']),
          'stream':True,'stream_options':{'include_usage':True}}
    return StreamingResponse(stream(req,route,body),media_type='text/event-stream')

async def stream(req: GenerateRequest, route: dict, body: dict):
    start=time.time()
    text=''
    tokens=0
    finish=None
    async with BUSY:
        try:
            async with httpx.AsyncClient(timeout=None) as client:
                async with client.stream('POST',f"http://localhost:{route['port']}/v1/completions",json=body) as r:
                    if r.status_code!=200:
                        yield event({'type':'error','message':f'Model server error {r.status_code}: {(await r.aread()).decode()[:300]}'})
                        return
                    async for line in r.aiter_lines():
                        if not line.startswith('data: ') or line=='data: [DONE]':
                            continue
                        chunk=json.loads(line[6:])
                        if chunk.get('usage'):
                            tokens=chunk['usage']['completion_tokens']
                        for choice in chunk.get('choices') or []:
                            if choice.get('text'):
                                text+=choice['text']
                                yield event({'type':'token','text':choice['text']})
                            if choice.get('finish_reason'):
                                finish=choice['finish_reason']
        except httpx.HTTPError as e:
            yield event({'type':'error','message':f'Model server error: {e}'})
            return
    kind=route['kind']
    final=final_answer(kind,text)
    first=get_first_answer(text) if kind in ('review','refine') else None
    latency_ms=int((time.time()-start)*1000)
    yield event({'type':'done',
                 'final_answer':final,
                 'first_answer':first,
                 'corrected':first is not None and final is not None and not same_answer(first,final),
                 'correct':is_correct(final,req.reference) if req.reference else None,
                 'fixed':(first is not None and not is_correct(first,req.reference) and is_correct(final,req.reference)) if req.reference else None,
                 'finished':finish=='stop',
                 'tokens':tokens,
                 'latency_ms':latency_ms})
    # one log line per request, without the question text
    print(json.dumps({'model':req.model,'question_chars':len(req.question),'tokens':tokens,'latency_ms':latency_ms,'finish':finish}),flush=True)

In [ ]:
json.dump(routes,open('/kaggle/working/routes.json','w'),indent=2)
os.environ['APP_API_KEY']=api_key
sys.path.insert(0,'/kaggle/working')
import router # the check below uses the router's own prompt and grading functions

## Check against the reported results
Runs the first `check_n` MATH-500 problems through every served model and compares the final answers with the eval outputs.
Expect at least 18 of 20 the same (small differences come from batching and LoRA vs merged weights). If a LoRA model is far off, use `MODE` merged.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset
def get_texts(m: str)->list:
    r=routes[m]
    prompts=[router.format_prompt(r['kind'],x['problem']) for x in check_set]
    out=requests.post(f"http://localhost:{r['port']}/v1/completions",json={'model':r['served'],'prompt':prompts,**router.sampling(r['kind'])}).json()
    return [c['text'] for c in sorted(out['choices'],key=lambda c:c['index'])]

if check_n>0:
    check_set=load_dataset("HuggingFaceH4/MATH-500",split="test").select(range(check_n))
    with ThreadPoolExecutor(len(routes)) as pool: # all models at once, vLLM batches them
        texts=dict(zip(routes,pool.map(get_texts,routes)))
    for m in routes:
        kind=routes[m]['kind']
        old=json.load(open(eval_files[MODELS[m]['eval']]))[:check_n]
        old_answers=[o['greedy_pred'] if kind=='base' else o['final_answer'] for o in old]
        new_answers=[router.final_answer(kind,t) for t in texts[m]]
        same=sum((a is None and b is None) or (a is not None and b is not None and router.same_answer(a,b)) for a,b in zip(new_answers,old_answers))
        right=sum(router.is_correct(a,x['answer']) for a,x in zip(new_answers,check_set))
        print(f'{m:14} same as eval: {same}/{check_n}   correct: {right}/{check_n}')

## Start the router and the public tunnel

In [ ]:
router_log=open('/kaggle/working/router.log','w')
router_proc=subprocess.Popen(['uvicorn','router:app','--host','0.0.0.0','--port','9000'],cwd='/kaggle/working',stdout=router_log,stderr=subprocess.STDOUT)
time.sleep(5)
print(requests.get('http://localhost:9000/health').json())
print("Without key:", requests.post('http://localhost:9000/generate',json={'model':list(routes)[0],'question':'1+1?'}).status_code) # expect 401

In [ ]:
# one short streamed answer through the router, as the chat app will see it
test_model=[m for m in routes if MODELS[m]['kind']=='review'][0] if any(MODELS[m]['kind']=='review' for m in routes) else list(routes)[0]
with requests.post('http://localhost:9000/generate',headers={'Authorization':'Bearer '+api_key},
                   json={'model':test_model,'question':'What is $2+3\\cdot 4$?','reference':'14'},stream=True) as r:
    events=[json.loads(l[6:]) for l in r.iter_lines(decode_unicode=True) if l.startswith('data: ')]
print(test_model)
print(''.join(e.get('text','') for e in events)[:1500])
print(events[-1])

In [ ]:
# Start the ngrok agent directly (pyngrok's connect(domain=...) is rejected by the current agent: "invalid endpoint configuration")
from pyngrok import conf, ngrok
ngrok.install_ngrok()
ngrok_path=conf.get_default().ngrok_path
public_url='https://'+ngrok_domain
def start_tunnel(flag: str):
    log=open('/kaggle/working/ngrok.log','w')
    return subprocess.Popen([ngrok_path,'http','9000',f'{flag}={public_url}','--authtoken',ngrok_token,'--log','stdout'],stdout=log,stderr=subprocess.STDOUT)
def tunnel_up(timeout: int=60)->bool:
    start=time.time()
    while time.time()-start<timeout:
        try:
            if requests.get(public_url+'/health',headers={'ngrok-skip-browser-warning':'1'},timeout=5).ok:
                return True
        except requests.exceptions.RequestException:
            pass
        time.sleep(3)
    return False
tunnel_proc=start_tunnel('--url')
if not tunnel_up():
    tunnel_proc.kill()
    tunnel_proc=start_tunnel('--domain') # older agents call the flag --domain
    assert tunnel_up(), open('/kaggle/working/ngrok.log').read()[-3000:]
print("Public URL:", public_url) # BACKEND_URL_A or BACKEND_URL_B on Vercel
print(requests.get(public_url+'/health',headers={'ngrok-skip-browser-warning':'1'}).json())

## Keep running
Keeps the session (and the site) alive for `RUN_HOURS` (set by push.ps1, at most Kaggle's 12-hour limit). To switch off earlier, cancel the run on Kaggle.

In [ ]:
deadline=time.time()+float(config.get('RUN_HOURS',11.5))*3600
while time.time()<deadline:
    alive={s['name']+f"@gpu{s['gpu']}":p.poll() is None for s,p in zip(servers,procs)}
    alive['router']=router_proc.poll() is None
    alive['ngrok']=tunnel_proc.poll() is None
    print(time.strftime('%H:%M'),alive,flush=True)
    if not all(alive.values()):
        print(open('/kaggle/working/router.log').read()[-3000:])
        break
    time.sleep(300)
print('Run time over, shutting down')